In [1]:
# Python Standard Library
import sys
from os import mkdir
from os.path import join
from os.path import isdir
from shutil import rmtree

# Community Modules
from tqdm import tqdm
import numpy as np
import pandas as pd
from pandas.testing import assert_frame_equal
from matplotlib import pyplot as plt

# My Modules
sys.path.insert(0, "../code")
import measure_signal as ms
from unpack_dataset import unpack_dataset

2026-03-20 16:11:10.803391: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-20 16:11:10.823542: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-20 16:11:10.829698: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-20 16:11:10.845581: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-20 16:11:12.656990: W tensorflow/compiler/tf2

In [2]:
DIR_DATA = "../data/forSNR/"

In [3]:
file_dataset = join(DIR_DATA, "dataset.parquet")
file_signal = join(DIR_DATA, "signal.parquet")
file_noise = join(DIR_DATA, "noise.parquet")

In [4]:
df_dataset = pd.read_parquet(file_dataset)
df_signal = pd.read_parquet(file_signal)
df_noise = pd.read_parquet(file_noise)

In [5]:
df_dataset = df_dataset.sort_values(by=["SN Subtype ID", "SNR"], ascending=[True, False]).reset_index(drop=True)
df_signal = df_signal.sort_values(by=["SN Subtype ID", "SNR"], ascending=[True, False]).reset_index(drop=True)
df_noise = df_noise.sort_values(by=["SN Subtype ID", "SNR"], ascending=[True, False]).reset_index(drop=True)

In [6]:
wvl, spectra_dataset, df_meta = unpack_dataset(df_dataset)
_1_wvl, spectra_signal, df_meta_signal = unpack_dataset(df_signal)
_2_wvl, spectra_noise, df_meta_noise = unpack_dataset(df_noise)
assert np.all(wvl == _1_wvl)
assert np.all(wvl == _2_wvl)
del _1_wvl, _2_wvl
assert_frame_equal(df_meta, df_meta_signal)
assert_frame_equal(df_meta, df_meta_noise)
del df_meta_signal, df_meta_noise

assert wvl.ndim == 1, (wvl.ndim, wvl.shape)
num_wvl = wvl.size
num_spectra = spectra_dataset.shape[0]

In [7]:
def plot_with_SNR_stuff(ax, specsnr):
    ax.set_xlim((4500, 7000))
    ax.axis("off")
    
    ax.plot(wvl, specsnr.spectrum, c="k", lw=1)
    ax.plot(wvl, specsnr.signal, c="tab:blue")

    vertical_midpoint = np.sum(ax.get_ylim()) / 2
    horizontal_text_location = ax.get_xlim()[0]
    ax.text(
        horizontal_text_location, vertical_midpoint,
        f"{specsnr.name}\n{specsnr.subtype}\n{specsnr.phase} days",
        ha="right", va="center")

    horizontal_text_location = ax.get_xlim()[1]
    ax.text(
        horizontal_text_location, vertical_midpoint,
        f"SNR = {specsnr.SNR:.2f}\nS = {specsnr.S:.2e}\nN = {specsnr.N:.2e}\n$\sigma = {specsnr.denoising_parameter}$",
        ha="left", va="center")

    ax.plot(specsnr.pc_wvl, specsnr.pseudo_cont, c="tab:green")

    # Color in the regions used to calculate the noise on the red and blue
    # shoulders.
    if specsnr.useBlu:
        ax.fill_between(
            specsnr.wvl[specsnr.blu_inds],
            y1=1,
            y2=0,
            color="tab:blue",
            alpha=0.25)
        ax.fill_between(
            specsnr.wvl[specsnr.blu_inds],
            y1=1,
            y2=0,
            color="tab:blue",
            alpha=0.25)

    if specsnr.useRed:
        ax.fill_between(
            specsnr.wvl[specsnr.red_inds],
            y1=1,
            y2=0,
            color="tab:red",
            alpha=0.25)
        ax.fill_between(
            specsnr.wvl[specsnr.red_inds],
            y1=1,
            y2=0,
            color="tab:red",
            alpha=0.25)
    return

In [8]:
subtypes_ID = df_dataset["SN Subtype ID"].unique()
subtypes_ID_to_str = dict(
    df_dataset.groupby(by="SN Subtype ID")["SN Subtype"].apply(
        lambda x: x.to_numpy()[0]))

In [9]:
DIR_FIGS = "../spectra_sparklines_SNRinfo"
if isdir(DIR_FIGS):
    rmtree(DIR_FIGS)
mkdir(DIR_FIGS)

for subtype_ID, subtype_str in subtypes_ID_to_str.items():
    dir_subtype_figs = join(DIR_FIGS, f"{subtype_ID:0>2}_{subtype_str}")
    if isdir(dir_subtype_figs):
        rmtree(dir_subtype_figs)
    mkdir(dir_subtype_figs)

In [10]:
for subtype_ID, subtype_str in subtypes_ID_to_str.items():
    num_sparklines_per_fig = 10
    
    dir_subtype_figs = join(DIR_FIGS, f"{subtype_ID:0>2}_{subtype_str}")
    
    mask = (df_meta["SN Subtype ID"] == subtype_ID)
    df_subtype = df_meta[mask].copy(deep=True).reset_index(drop=True)
    subtype_spectra = spectra_dataset[mask]
    assert mask.sum() == df_subtype.shape[0]
    assert mask.sum() == subtype_spectra.shape[0]
    subtype_num_spectra = mask.sum()
    print(subtype_ID, subtype_str, subtype_num_spectra)
    
    for i in tqdm(range(0, subtype_num_spectra, num_sparklines_per_fig)):
        if not (i + num_sparklines_per_fig <= subtype_num_spectra):
            num_sparklines_per_fig = subtype_num_spectra - i
        
        fig, axes = plt.subplots(
            ncols=1,
            nrows=num_sparklines_per_fig,
            figsize=(6, num_sparklines_per_fig))

        for j in range(num_sparklines_per_fig):
            if num_sparklines_per_fig == 1:
                ax = axes
            else:
                ax = axes[j]
            
            subtype_df_meta_ij = df_subtype.loc[i+j]
            spectrum_ij = subtype_spectra[i+j]
            
            assert subtype_df_meta_ij.ndim == 1
            assert spectrum_ij.ndim == 1
            
            specsnr = ms.SpectrumSNR(
                subtype_df_meta_ij["SN Name"],
                subtype_df_meta_ij["SN Subtype"],
                subtype_df_meta_ij["Spectrum Phase"],
                wvl,
                spectrum_ij,
            )
            specsnr.execute_algorithm(subtype_df_meta_ij)
            
            plot_with_SNR_stuff(ax, specsnr)
        
        fig.tight_layout()
        fig.savefig(join(dir_subtype_figs, f"{i:0>5}"))
        plt.close()

0 Ia-norm 2034


100%|██████████| 204/204 [03:11<00:00,  1.06it/s]


1 Ia-91T 339


100%|██████████| 34/34 [00:30<00:00,  1.11it/s]


2 Ia-91bg 222


...

100%|██████████| 18/18 [00:18<00:00,  1.05s/it]


8 Ic-broad 210


100%|██████████| 21/21 [00:19<00:00,  1.10it/s]


9 IIP 100


100%|██████████| 10/10 [00:09<00:00,  1.10it/s]
